In [ ]:
import cv2
from google.colab.patches import cv2_imshow
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import griddata

#1

In [ ]:
patronD = cv2.imread("/content/difracción3Fresnel.jpg" , 0)
patronD = cv2.resize(patronD, (960, 960))
#patronD_umbral = cv2.adaptiveThreshold(patronD, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)
#patronD = cv2.bitwise_not(patronD_umbral)
ret, patronD_umbral = cv2.threshold(patronD, 100, 255, cv2.THRESH_BINARY)
cv2_imshow(patronD_umbral)

In [ ]:
plt.imshow(patronD_umbral, cmap='gray')
plt.gca().invert_yaxis()

In [ ]:
X , Y = [], []
for x in range(patronD_umbral.shape[0]):
  for y in range(patronD_umbral.shape[1]):
    if patronD_umbral[x, y] == 255:
      X.append(x)
      Y.append(y)


plt.figure(figsize=(50,50))
plt.plot(X,Y, "o", c = "r")
plt.locator_params(axis='x', nbins=100)
plt.locator_params(axis='y', nbins=100)
plt.grid()

In [ ]:
#cambio de eje
X_new = np.array(X)-485
Y_new = np.array(Y)



plt.figure(figsize=(50,50))
plt.plot(X_new,Y_new, "o", c = "r")
plt.locator_params(axis='x', nbins=100)
plt.locator_params(axis='y', nbins=100)
plt.grid()

In [ ]:
X_real = X_new *0.0001107692
Y_real = Y_new *0.0001107692

plt.figure(figsize=(50,50))
plt.plot(X_new,Y_new, "o", c = "r")
plt.locator_params(axis='x', nbins=100)
plt.locator_params(axis='y', nbins=100)
plt.xlabel("X (metros)")
plt.ylabel("Y (metros)")
plt.grid()

In [ ]:
L = 2000
N = len(X_real)
dx = L / N
w = 1.8e-6
lam = 0.5 * 1e-6
k = 2 * np.pi / lam
z = 2000
j = complex(0,1)

In [ ]:
U2 = [X_real , Y_real]

r = np.sqrt(z**2 + X_real**2 + Y_real**2)
h = ( z / j*lam*(r**2) ) * np.exp(j*k*r)

TfU2 = np.fft.fft(U2)
TfH = np.fft.fft(h)

U1 = np.fft.ifft(TfU2 / TfH)

In [ ]:
#plt.figure(figsize=(50,50))
plt.plot(U1[0] , U1[1] , "o" , c= "black")
plt.title('La rendija supuestamente :)')
plt.locator_params(axis='x', nbins=100)
plt.locator_params(axis='y', nbins=100)
plt.xlabel("X (metros)")
plt.ylabel("Y (metros)")
#plt.xlim(-1e-12 , 1e-12)
#plt.ylim(-1e-12 , 1e-12)
plt.grid()
plt.show()

#2

In [ ]:
#visualizacion de la imagen para verificacion
def visualizacion(imagen):
  patronD = cv2.imread(imagen , 0)
  patronD = cv2.resize(patronD, (960, 960))
  #patronD_umbral = cv2.adaptiveThreshold(patronD, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)
  #patronD = cv2.bitwise_not(patronD_umbral)
  ret, patronD_umbral = cv2.threshold(patronD, 80, 255, cv2.THRESH_BINARY)

  plt.figure(figsize=(50,50))
  plt.imshow(patronD_umbral, cmap='gray')
  plt.locator_params(axis='x', nbins=100)
  plt.locator_params(axis='y', nbins=100)
  plt.xlabel("X (pixeles)")
  plt.ylabel("Y (pixeles)")
  plt.grid()
  plt.show()
  plt.gca().invert_yaxis()

  return patronD_umbral

In [ ]:
def difraccion(imagen , relacionpx , z, w , campo): #campo = fresnel o fraun

  #PARTE I. MONTAR IMAGEN
  patronD = cv2.imread(imagen , 0)
  patronD = cv2.resize(patronD, (960, 960))
  #patronD_umbral = cv2.adaptiveThreshold(patronD, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)
  #patronD = cv2.bitwise_not(patronD_umbral)
  ret, patronD_umbral = cv2.threshold(patronD, 100, 255, cv2.THRESH_BINARY)


  #PARTE II. DEFINIR LISTAS X y Y
  X , Y = [], []    #filtral x,y talque definan elpatron de difraccion
  for x in range(patronD_umbral.shape[0]):
    for y in range(patronD_umbral.shape[1]):
      if patronD_umbral[x, y] == 255:
        X.append(x)
        Y.append(y)          #obencion de los puntos x,y en terminos de los pixeles

  X_new = np.array(X)-485   #traslacion del eje x
  Y_new = np.array(Y)

  X_real = X_new *relacionpx  #relacion entre pixeles y distancia real
  Y_real = Y_new *relacionpx

  #PARTE III.ENCONTRAR U1
  #definicion de parametros
  L = z #L = distancia z
  N = len(X_real)
  dx = L / N
  lam = 0.5 * 1e-6
  k = 2 * np.pi / lam
  j = complex(0,1)

  U2 = [X_real , Y_real]
  r = np.sqrt(z**2 + X_real**2 + Y_real**2)
  h = ( z / j*lam*(r**2) ) * np.exp(j*k*r)
  TfU2 = np.fft.fft(U2)
  TfH = np.fft.fft(h)
  U1 = np.fft.ifft(TfU2 / TfH)

  plt.figure(figsize=(50,50))
  plt.plot(U1[0] , U1[1] , "o" , c= "black")
  plt.title('La rendija supuestamente :)')
  plt.locator_params(axis='x', nbins=100)
  plt.locator_params(axis='y', nbins=100)
  plt.xlabel("X (metros)")
  plt.ylabel("Y (metros)")
  #plt.xlim(-0.005 , 0.005)
  #plt.ylim(-0.005 , 0.005)
  plt.grid()
  plt.show()

  return

